# Qwen3-0.6B DPO Fine-Tuning Notebook

这份 notebook 使用本地缓存的 `Qwen/Qwen3-0.6B` 模型，基于当前目录下的 `DPO_data.json` 进行 DPO（Direct Preference Optimization）偏好对齐训练。


## 1. 安装依赖

这一部分安装 DPO 训练所需的基本库：`transformers` 负责模型加载与生成，`trl` 提供 `DPOTrainer`，`peft` 负责 LoRA 参数高效微调，`datasets` 负责数据集处理。

从流程上看，DPO 会优化一个偏好目标，而不是单纯做 next-token 监督学习；但在正式进入优化之前，先要把工具链准备好。


In [ ]:
%pip install -q transformers datasets peft accelerate trl sentencepiece safetensors typing-inspection


## 1.1 Compatibility Bootstrap

This cell isolates the notebook from user-level site-packages and blocks optional quantization packages that are not needed for this DPO run.

It is mainly used to avoid two Windows environment issues: `AppData/Roaming/Python/.../site-packages` shadowing the Conda environment, and `awq` / `bitsandbytes` being auto-detected even though they are not required here.


In [1]:
import os
import sys
import site
import importlib.util

os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
user_site = site.getusersitepackages()
if user_site in sys.path:
    sys.path.remove(user_site)

# If this kernel was used before, clear conflicting modules so later imports use the cleaned sys.path.
for module_name in list(sys.modules):
    if module_name.split(".")[0] in {"transformers", "peft", "trl", "datasets", "awq", "autoawq", "bitsandbytes"}:
        sys.modules.pop(module_name, None)

_original_find_spec = importlib.util.find_spec
blocked_optional_packages = {"awq", "autoawq", "bitsandbytes", "tensorflow", "keras"}

def patched_find_spec(name, package=None):
    root_name = name.split(".")[0]
    if root_name in blocked_optional_packages:
        return None
    return _original_find_spec(name, package)

importlib.util.find_spec = patched_find_spec

print("PYTHONNOUSERSITE =", os.environ["PYTHONNOUSERSITE"])
print("user site removed =", user_site not in sys.path)
print("blocked optional packages =", sorted(blocked_optional_packages))


PYTHONNOUSERSITE = 1
user site removed = True
blocked optional packages = ['autoawq', 'awq', 'bitsandbytes', 'keras', 'tensorflow']


## 2. 定位本地模型与偏好数据

这一段负责定义基座模型、偏好数据文件、训练输出目录，并检查 `DPO_data.json` 的基本结构。

对 DPO 来说，一条样本可以记为：

$$
\mathcal{D} = \{(x_i, y_i^{+}, y_i^{-})\}_{i=1}^{N}
$$

其中 $x_i$ 是 prompt，$y_i^{+}$ 是更受偏好的回答，$y_i^{-}$ 是较差的回答。


In [2]:
from pathlib import Path
import json
import os

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, TaskType
from trl import DPOConfig, DPOTrainer

BASE_MODEL_DIR = Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen3-0.6B" / "snapshots" / "c1899de289a04d12100db370d81485cdf75e47ca"
DATA_PATH = Path.cwd() / "DPO_data.json"
OUTPUT_DIR = Path.cwd() / "qwen3_dpo_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_model_files = ["config.json", "model.safetensors", "tokenizer.json"]
print("BASE_MODEL_DIR =", BASE_MODEL_DIR)
print("DATA_PATH =", DATA_PATH)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("model exists =", BASE_MODEL_DIR.exists())
print("data exists =", DATA_PATH.exists())
for name in required_model_files:
    print(f"{name}:", (BASE_MODEL_DIR / name).exists())

raw_records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
print("num preference pairs =", len(raw_records))
print("sample keys =", list(raw_records[0].keys()))
print("sample prompt =", raw_records[0]["prompt"])


c:\Users\z5364\.conda\envs\normal\lib\site-packages\wandb\sdk\launch\builder\build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


BASE_MODEL_DIR = C:\Users\z5364\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B\snapshots\c1899de289a04d12100db370d81485cdf75e47ca
DATA_PATH = c:\Users\z5364\Desktop\0414组会\0414组会\DPO_data.json
OUTPUT_DIR = c:\Users\z5364\Desktop\0414组会\0414组会\qwen3_dpo_output
model exists = True
data exists = True
config.json: True
model.safetensors: True
tokenizer.json: True
num preference pairs = 12
sample keys = ['prompt', 'chosen', 'rejected']
sample prompt = 什么是大语言模型？


## 3. 加载 tokenizer、policy model 与 reference model

DPO 的核心是比较“当前策略模型”和“参考模型”对 chosen / rejected 的相对偏好。这里我们使用同一个基座模型初始化两份权重：

$$
\pi_{\theta} \quad \text{and} \quad \pi_{\mathrm{ref}}
$$

其中 `\pi_\theta` 会在 LoRA 参数上被更新，`\pi_{ref}` 保持不变，用作稳定的对照项。


In [3]:
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == "cuda" else torch.float32)

torch.set_num_threads(max(1, (os.cpu_count() or 1) // 2))
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR), trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_DIR),
    dtype=DTYPE,
    trust_remote_code=False,
)

model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

desired_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
linear_suffixes = sorted({name.split(".")[-1] for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)})
target_modules = [name for name in desired_targets if name in linear_suffixes]
print("Loaded models with device preference:", DEVICE, "dtype=", DTYPE)
print("LoRA target modules =", target_modules)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules,
)


Loaded models with device preference: cuda dtype= torch.bfloat16
LoRA target modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 4. 构造 DPO 训练样本

这一段会把原始 `prompt / chosen / rejected` 格式转成 `DPOTrainer` 可直接使用的形式。对于 chat 模型，prompt 需要先被组装成对话模板，然后再接上 chosen 和 rejected 两个候选回答。

DPO 的直观目标是让模型更偏向 chosen，更远离 rejected。常见写法为：

$$
\mathcal{L}_{\mathrm{DPO}} = - \log \sigma \left( \beta \left[ \log \frac{\pi_{\theta}(y^{+}|x)}{\pi_{\mathrm{ref}}(y^{+}|x)} - \log \frac{\pi_{\theta}(y^{-}|x)}{\pi_{\mathrm{ref}}(y^{-}|x)} \right] \right)
$$

其中 `\beta` 用来控制偏好信号的强度。


In [4]:
SYSTEM_PROMPT = "\u4f60\u662f\u4e00\u4e2a\u4e50\u4e8e\u52a9\u4eba\u7684\u4e2d\u6587\u52a9\u624b\uff0c\u8bf7\u6839\u636e\u7528\u6237\u7684\u95ee题\u7ed9\u51fa\u6e05\u6670\u3001\u51c6\u786e\u3001\u6709\u6761\u7406\u7684\u56de\u7b54\u3002"
MAX_LENGTH = 384

def normalize_text(text):
    return str(text).strip()

def build_chat_prompt(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": normalize_text(user_prompt)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=True,
        tokenize=False,
    )

def load_dpo_records(path: Path):
    records = json.loads(path.read_text(encoding="utf-8"))
    cleaned = []
    for row in records:
        if not isinstance(row, dict):
            continue
        prompt = normalize_text(row.get("prompt", ""))
        chosen = normalize_text(row.get("chosen", ""))
        rejected = normalize_text(row.get("rejected", ""))
        if not prompt or not chosen or not rejected:
            continue
        cleaned.append({
            "prompt": build_chat_prompt(prompt),
            "chosen": chosen,
            "rejected": rejected,
        })
    return cleaned

formatted_records = load_dpo_records(DATA_PATH)
dataset = Dataset.from_list(formatted_records)
split_dataset = dataset.train_test_split(test_size=min(2, max(1, len(dataset) // 5)), seed=42, shuffle=True)
train_ds = split_dataset["train"]
eval_ds = split_dataset["test"]

print("formatted pairs =", len(dataset))
print("train size =", len(train_ds))
print("eval size =", len(eval_ds))
print("sample formatted prompt =\n", train_ds[0]["prompt"])
print("sample chosen =\n", train_ds[0]["chosen"])
print("sample rejected =\n", train_ds[0]["rejected"])


formatted pairs = 12
train size = 10
eval size = 2
sample formatted prompt =
 <|im_start|>system
你是一个乐于助人的中文助手，请根据用户的问题给出清晰、准确、有条理的回答。<|im_end|>
<|im_start|>user
什么是损失函数？<|im_end|>
<|im_start|>assistant

sample chosen =
 好问题！损失函数可以看作模型的“成绩评定器”，它用来衡量模型当前预测结果到底有多差。预测得越准，损失就越小；偏差越大，损失就越大。训练模型的过程，本质上就是不断调整参数，让损失函数的值尽可能下降。你完全可以把它理解成模型学习过程中的“方向盘”——它告诉模型哪里做错了，以及应该往哪个方向改进！
sample rejected =
 损失函数用于衡量模型预测结果与真实值之间的差异，训练目标通常是最小化损失。


## 5. 设定 DPO 训练参数与 Trainer

这一部分用 `DPOConfig` 定义训练参数，包括 batch size、学习率、`beta`、最大序列长度和截断策略。

由于 `DPO_data.json` 只有 12 条样本，这里的配置更偏向「演示可运行」而不是「正式大规模对齐训练」。如果你后面换成更大的 preference 数据集，可以直接在这里放大参数。


In [5]:
training_args = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=1,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    max_length=MAX_LENGTH,
    truncation_mode="keep_start",
    beta=0.1,
    bf16=DEVICE == "cuda" and torch.cuda.is_bf16_supported(),
    fp16=DEVICE == "cuda" and not torch.cuda.is_bf16_supported(),
    tf32=DEVICE == "cuda",
    report_to="none",
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

def count_parameters(model_obj):
    total_params = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)
    ratio = 100 * trainable_params / total_params if total_params else 0.0
    print(f"total params = {total_params:,}")
    print(f"trainable params = {trainable_params:,}")
    print(f"trainable ratio = {ratio:.4f}%")

count_parameters(trainer.model)
if hasattr(trainer.model, "print_trainable_parameters"):
    trainer.model.print_trainable_parameters()

print("Trainer ready")


Extracting prompt in train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

total params = 601,096,192
trainable params = 5,046,272
trainable ratio = 0.8395%
trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395
Trainer ready


## 6. 启动 DPO 训练

这一段会正式启动 `trainer.train()`。从优化角度看，我们在更新的并不是整个基座模型，而是 LoRA 附加的低秩参数，即：

$$
\theta' = \theta + \Delta \theta_{\mathrm{LoRA}}
$$

这样可以在保持显存占用较低的同时，学到更符合偏好数据的输出倾向。


In [ ]:
train_result = trainer.train()
print(train_result)


## 7. 保存 LoRA Adapter，合并权重，并进行测试生成

训练结束后，可以只保存 adapter，也可以把 LoRA 增量合并回基座模型，得到一个更方便部署的单体模型：

$$
W_{\mathrm{merged}} = W + BA
$$

最后再用一条示例 prompt 做推理，检查训练后的模型是否能正常生成。


In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter to", ADAPTER_DIR)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print("Saved merged model to", MERGED_DIR)

def chat(model_obj, prompt: str, system: str = SYSTEM_PROMPT, max_new_tokens: int = 128) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(merged_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = merged_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(chat(merged_model, "请用一句话解释什么是 DPO 训练。"))
